# Od podatkov do podatkovnih tabel

Podatke za analizo lahko zberemo na različne načine. Pripravimo jih lahko iz lastnih meritev in jih v knjižnici `pandas` shranimo v zaporedja in podatkovne tabele. Lahko jih pridobimo tudi v obliki preglednic Excel iz statističnih virov, kot so [podatkovna baza SiStat](https://pxweb.stat.si/) Statističnega urada Republike Slovenije, [podatkovna baza Eurostat](https://ec.europa.eu/eurostat/data/database) evropskega statističnega urada in [podatkovna baza UNdata](https://data.un.org/) Organizacije združenih narodov. Bogat vir tabel je tudi spletna enciklopedija [Wikipedia](https://www.wikipedia.org/), na primer [članek s podatki o bruto domačem proizvodu držav](https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)).

Knjižnica `pandas` ponuja funkcije za branje preglednic Excel in tabel s spletnih strani. Sam uvoz je zato pogosto preprost. Več dela zahteva urejanje, saj imajo podatki iz različnih virov različno **strukturo**.

## Uvoz neurejenih podatkov

Profesorica, ki na FMF izvaja dvosemestrski predmet, si je v datoteki [`rezultati-kolokvijev.xlsx`](https://kt.ijs.si/~ljupco/lectures/papvp-2324/rezultati-kolokvijev.xlsx) zapisovala rezultate kolokvijev v prvem in drugem semestru. Prebrali bomo vse liste preglednice in podatke uredili v tabeli za analizo.

Preglednice Excel uvozimo s funkcijo [`read_excel`](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html). Spoznajmo njena prva dva argumenta:

  * `io` določa spletni naslov (URL) ali pot do datoteke. Argument nima privzete vrednosti.
  * `sheet_name` določa ime lista ali njegov položaj, štet od 0. Podamo lahko tudi seznam imen oziroma položajev. Vrednost `None` zahteva uvoz vseh listov v slovar podatkovnih tabel. Privzeta vrednost `0` pomeni prvi list.

V našem primeru potrebujemo vse liste:

In [1]:
import pandas as pd

vse_pt = pd.read_excel(
    "https://kt.ijs.si/~ljupco/lectures/papvp-2324/rezultati-kolokvijev.xlsx",
    sheet_name=None
)
print(vse_pt)

{'prvi kolokvij':       ime  telefon spol in starost  prvi semester  drugi semester
0     Ana      431            ž-19             80              97
1  Branko      720            m-20             78              74
2   David      141            m-19             75              68
3     Eva      210            ž-20             63              82
4   Franc      592            m-21             95              82, 'drugi kolokvij':       ime  telefon spol in starost  prvi semester  drugi semester
0     Ana      431            ž-19             82              95
1  Cvetka      761            ž-21             63              87
2   David      141            m-19             92              81
3     Eva      210            ž-20             82              63
4   Franc      592            m-21             99              54}


Rezultat je slovar z dvema elementoma. Ključa sta imeni listov (`prvi kolokvij` in `drugi kolokvij`), vrednosti pa podatkovni tabeli z ustreznima listoma.

Podatke iz datotek CSV (angl. _comma-separated values_, vrednosti, ločene z vejicami) uvozimo s funkcijo [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).

Funkcija [`concat`](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) omogoča združevanje tabel po vrsticah. Če bi obe tabeli takoj združili, bi izgubili podatek, kateremu kolokviju pripada posamezna vrstica:

In [2]:
pt = pd.concat(vse_pt.values())
print(pt)

      ime  telefon spol in starost  prvi semester  drugi semester
0     Ana      431            ž-19             80              97
1  Branko      720            m-20             78              74
2   David      141            m-19             75              68
3     Eva      210            ž-20             63              82
4   Franc      592            m-21             95              82
0     Ana      431            ž-19             82              95
1  Cvetka      761            ž-21             63              87
2   David      141            m-19             92              81
3     Eva      210            ž-20             82              63
4   Franc      592            m-21             99              54


Pred združevanjem zato vsaki tabeli dodamo stolpec `kolokvij`, ki vsebuje ime lista. Za ime spremenljivke uporabimo `ime_lista`, da ne prekrijemo vgrajenega imena `list`:

In [3]:
for ime_lista in vse_pt:
    vse_pt[ime_lista]["kolokvij"] = ime_lista
pt = pd.concat(vse_pt.values())
print(pt)

      ime  telefon spol in starost  prvi semester  drugi semester  \
0     Ana      431            ž-19             80              97   
1  Branko      720            m-20             78              74   
2   David      141            m-19             75              68   
3     Eva      210            ž-20             63              82   
4   Franc      592            m-21             95              82   
0     Ana      431            ž-19             82              95   
1  Cvetka      761            ž-21             63              87   
2   David      141            m-19             92              81   
3     Eva      210            ž-20             82              63   
4   Franc      592            m-21             99              54   

         kolokvij  
0   prvi kolokvij  
1   prvi kolokvij  
2   prvi kolokvij  
3   prvi kolokvij  
4   prvi kolokvij  
0  drugi kolokvij  
1  drugi kolokvij  
2  drugi kolokvij  
3  drugi kolokvij  
4  drugi kolokvij  


Podatke smo uvozili, pred analizo pa jih moramo še urediti. Tabela ima nekaj pomanjkljivosti:

  * Vrednosti spremenljivk `spol` in `starost` so združene v stolpcu `spol in starost`.

  * Osnovni podatki o študentih se po nepotrebnem ponovijo dvakrat.

  * Podatek o zaporedni številki kolokvija je podan kot vrednost stolpca, podatek o zaporedni številki semestra pa kot ime stolpca.

Pred odpravljanjem teh težav spoznajmo pojem _urejenih podatkov_.

## Urejeni podatki (angl. _tidy data_)

Dogovor o urejenih podatkih določa **tri preprosta pravila za podatkovne tabele**.

  1. Vsak tip opazovane enote tvori svojo podatkovno tabelo.

  1. Vsaka opazovana enota tvori vrstico.

  1. Vsaka lastnost opazovanih enot je spremenljivka, ki tvori stolpec.

Pri načrtovanju tabel se najprej vprašamo: **Katere tipe opazovanih enot vsebujejo podatki?** En del podatkov opisuje študente, drugi pa njihove dosežke na posameznih kolokvijih. Podatke zato ločimo v dve tabeli:

  * `studenti` z osnovnimi podatki o študentih, ki niso vezani na njihove dosežke na kolokvijih; in
  
  * `studenti_ocene` s podatki o dosežkih študentov na kolokvijih.
  
Drugo in tretje pravilo nam pomagata pri premisleku o vsebini posameznih tabel. Vsaka vrstica tabele `studenti` bo vsebovala osnovne podatke o enem študentu. Tabela bo zato imela šest vrstic. Naši podatki iz preglednice vsebujejo štiri osnovne podatke o vsakem študentu: ime, telefon, spol in starost. Ta seznam določa opazovane spremenljivke, zato bo tabela imela štiri ustrezno poimenovane stolpce. Prvi dve vrstici tabele `studenti` bosta:

| ime | telefon | spol | starost |
|:--|--:|:--:|--:|
| Ana | 431 | ž | 19 |   
| Branko | 720 | m | 20 |

Spremenljivke v zgornji tabeli lahko razdelimo v dve skupini. V prvi skupini je spremenljivka `ime`, ki je **fiksna spremenljivka**, katere vrednost določa opazovano enoto, v našem primeru posameznega študenta. Imenujemo jo tudi **dimenzijska spremenljivka**. Druge spremenljivke sodijo v skupino **merjenih spremenljivk**: njihove vrednosti določimo z opazovanjem ali merjenjem lastnosti študentov. Imenujemo jih tudi **vrednostne spremenljivke**.

Pri strukturi druge tabele `studenti_ocene` je možnosti več. Ena možnost je, da bi vsaka vrstica ustrezala enemu študentu. Za vsakega študenta bi opazovali štiri spremenljivke, ki ustrezajo posameznim kolokvijem, dvema kolokvijema v prvem in dvema v drugem semestru. Za analizo teh dosežkov bomo raje uporabili obliko, v kateri vsaka vrstica ustreza dosežku na enem kolokviju.

Dogovor o urejenih podatkih je nastal v okolju jezika R in ga uporabljajo paketi zbirke [`tidyverse`](https://www.tidyverse.org/). Uporaben je tudi pri analizi s knjižnico `pandas`. Več o njem lahko preberemo v članku Wickham, H. (2014). [Tidy Data](https://vita.had.co.nz/papers/tidy-data.html). _Journal of Statistical Software, 59(10)_, 1–23. DOI: [10.18637/jss.v059.i10](https://doi.org/10.18637/jss.v059.i10). Koristno branje je tudi članek [Data set](https://en.wikipedia.org/wiki/Data_set) v Wikipediji.

Predlog s štirimi stolpci za rezultate posameznega študenta ne ustreza izbrani opazovani enoti: dosežku študenta na določenem kolokviju v določenem semestru. V izvirni preglednici je na primer `prvi semester` ime stolpca, čeprav je `prvi` vrednost spremenljivke `semester`. Če upoštevamo to pravilo, potem bo imela podatkovna tabela z dosežki študentov štiri spremenljivke in ustrezne stolpce `ime`, `semester`, `kolokvij` in `rezultat`. Poglejmo prvih nekaj vrstic take tabele:

| ime | semester | kolokvij | rezultat |
|:--|--:|--:|--:|
| Ana | 1 | 1 | 80 |
| Ana | 2 | 1 | 97 |
| Ana | 1 | 2 | 82 |
| Ana | 2 | 2 | 95 |
| Branko | 1 | 1 | 78 |
| Branko | 2 | 1 | 74 |

V tem primeru imamo opravka s tremi dimenzijskimi spremenljivkami, ki skupaj določajo dosežek na konkretnem kolokviju (ime študenta, semester, zaporedna številka kolokvija v semestru), in eno vrednostno spremenljivko – rezultat, ki ga je študent dosegel na kolokviju, določenem z vrednostmi dimenzijskih spremenljivk.

Na tem mestu lahko tudi bolj formalno, matematično opredelimo osnovne pojme v tako sestavljeni tabeli:

  * **Podatkovna tabela** zapisuje podatke o opazovanih enotah $o$. Množico opazovanih enot označimo z $O$. Za vsako opazovano enoto torej velja $o \in O$.

  * **Spremenljivka** $X$ je preslikava $X\colon O \to D_X$, kjer je $D_X$ domena spremenljivke, tj. množica vseh njenih možnih vrednosti. Vrednost spremenljivke $\textrm{v}_X(o)$ za podano enoto $o$ dobimo torej kot rezultat preslikave (sliko enote $o$), $X\colon o \mapsto \textrm{v}_X(o)$, pri čemer velja $\textrm{v}_X(o) \in D_X$.

  * Podatkovna tabela vsebuje $p$ spremenljivk: $d$ dimenzijskih in $p-d$ vrednostnih. Dimenzijske spremenljivke običajno ustrezajo prvim $d$ stolpcem podatkovne tabele $X_1, X_2, \dots, X_d$, zadnjih $p-d$ stolpcev pa ustreza vrednostnim spremenljivkam $X_{d+1}, \dots, X_{p}$.

  * Če kombinacija dimenzijskih vrednosti enolično določa opazovano enoto, lahko tabelo razumemo kot tabelarično predstavitev preslikave $f\colon D \to D_{d+1} \times \dots \times D_p$, pri čemer je vsak $D_i$ domena ustrezne spremenljivke $X_i$. Množica $D \subseteq D_1 \times \dots \times D_d$ vsebuje opazovane kombinacije dimenzijskih vrednosti. Dogovor o urejenih podatkih sam po sebi ne zahteva vrstic za vse možne kombinacije.

  * Vsaka vrstica podatkovne tabele podaja vrednosti $\textrm{v}_{X_i}(o)$ vseh spremenljivk $X_1, X_2, \dots X_p$ za opazovano enoto $o$.